# 高速版：1問（M2）で創薬標的の可能性を採点する

3つの条件を1つの質問にまとめ、「**どれか1つに当てはまるか**」を Yes/No で聞きます。1遺伝子あたり1問なので、6問版より大幅に速く、全遺伝子を両順（Yes/No と No/Yes）で聞いても 100 遺伝子で約5分（txgemma-9b Q6_K、Mac）です。

## 質問 M2（3条件の OR）

| 条件 | 中身 | 役割 |
|---|---|---|
| (a) | 遺伝子 {gene} を阻害または活性化すると、{disease} を治療する、または上に挙げた症状の少なくとも1つを改善する、という仮説を構築できる | 広く拾う（旧 H3 を症状改善まで拡張） |
| (b) | この遺伝子が原因経路に含まれていなくても、並行・拮抗する経路を通じて異常な過程を打ち消す・補える | 代償経路の救済（旧 R1。例：軟骨無形成症の NPR2・GHR） |
| (c) | この遺伝子自身の役割（基質・リガンド・シグナル経路・細胞・回路）が、記載した機構と正確に一致する（同じ遺伝子ファミリーでも似て非なる役割なら不可） | 絞り込む（旧 B1） |

**遺伝子名・病名の書き方は条件ごとに違います（意図的です）。** `scripts/yesno_question_variants.py` での実機検証の結果、(a) は遺伝子名と病名を書く方が、(b) は遺伝子名を書かず（this gene）病名を書く方が、(c) はどちらも書かず上の箇条書きを参照させる方が良かったためです。書き方をそろえた版（R1b / B1b / H3c / R1c）はどれも成績が下がりました。

## 実機検証の結果（txgemma-9b-chat Q6_K、5疾患 × 100遺伝子、両順）

| 方式 | 問数 | AUC 既知 vs ダミー（平均/最低） | AUC 既知 vs その他 | AUC 候補 vs ダミー | ダミーの Yes 率 |
|---|---|---|---|---|---|
| 6問（A1〜B4 を別々に聞いて平均） | 6 | 0.945 / 0.811 | 0.823 | 0.862 | — |
| 3問（H3・R1・B1 を別々に聞いて平均） | 3 | 0.959 / 0.905 | 0.859 | 0.888 | 0.25 |
| M1（(a) が「治療薬が治療する」の旧版） | 1 | 0.961 / 0.887 | 0.843 | 0.896 | 0.64 |
| **M2（この notebook）** | **1** | **0.986 / 0.971** | **0.872** | 0.894 | 0.43 |

- M2 は M1 の条件 (a) だけを「阻害または活性化で、治療する、または症状の少なくとも1つを改善する」に広げた版です（(b)(c) は同じ文面）。**5疾患すべてで AUC 既知 vs その他が M1 以上**になり、1問で3問版も上回りました。
- 条件を広げたのに、**ダミーの Yes 率は 5疾患すべてで下がりました**（0.64 → 0.43）。「阻害または活性化」「上に挙げた症状の少なくとも1つ」と具体的にしたことで、何でも仮説が立つという甘い Yes が減ったと考えられます。シスチン尿症の同じファミリーのダミーとの AUC は M1 と同じ（0.722）で、ファミリーの後光効果は悪化していません。
- 既知遺伝子の順位は、前立腺がんの FOLH1（87 → 49 位）、RA の NR3C1（63 → 49）・DHFR（78 → 67）、統合失調症の CHRM1/CHRM4（30/33 → 22/26）などが上がり、大きく下がったものはありません（最大 6 位）。
- **注意：M2 でもダミーに Yes が出ます（中央値 0.43、シスチン尿症では 0.93）。** **p_yes を「0.5 以上なら標的」のような絶対基準には使わず、順位付けに使ってください**（既知・候補はダミーより上に分かれています）。
- どの条件で Yes になったかは分かりません。理由を知りたい遺伝子は、`scripts/yesno_question_variants.py --qids H3 R1 B1` で3条件を別々に聞いてください。
- 質問では救えない既知遺伝子があります（RA の DHODH・MS4A1・CD80/86 など。作用が病気に特異的でない標的）。改善するなら質問文より病気の説明文（`DISEASE_INFO`）の側です。
- 5遺伝子から選ぶ相対評価（ノートブック 04 方式）でも同じ3条件を試しましたが（M1r / M2r）、Yes/No より少し劣りました（`scripts/rank_variants.py`）。

## 前提
- 本命は **`~/llm/models` の GGUF を llama-cpp-python で直接動かす**経路です（前置きの KV キャッシュを保存・復元して、遺伝子ごとに処理するのは遺伝子名と質問だけ）。Ollama では前置きの再処理が避けられないため遅くなります。
- モデルが無い環境ではモック（擬似乱数）で動きます。数値に意味はありません。
- 病気の情報は「症状 → 臓器・細胞・機能」で書き、分子名・原因遺伝子名は書きません（ノートブック 02 と同じルール）。
- 6問版（A1〜B4・PMI・H1/H2 診断）は git の履歴に残っています（コミット 0459887 以前の本ファイル）。

### このセルがすること：準備

In [1]:
import os, re, math, glob, json, time, random, hashlib, urllib.request
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
ROOT = os.path.abspath("..")
print("ready:", ROOT)

ready: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02


## 疾患の選択

### このセルがすること：`data/diseases.json` の登録疾患から1つ選び、病名・症状の箇条書き・遺伝子リストを読み込む
- `DISEASE_KEY` を変えるだけで切り替わります：`ra` / `scz` / `cystinuria` / `prostate_cancer` / `achondroplasia`。
- 箇条書きは「症状 → 臓器・細胞・機能」で書かれ、分子名・原因遺伝子名を含みません（テストで HGNC 記号との照合済み）。
- 遺伝子リストは `data/genes/<疾患>_set100.tsv`。RA 以外はダミーに SLC トランスポーターを多数含みます（シスチン尿症は原因遺伝子も SLC なので厳しい検証になります）。

In [2]:
DISEASE_KEY = "ra"                    # "ra" / "scz" / "cystinuria" / "prostate_cancer" / "achondroplasia"
GENE_SET = "set100"                   # "set100" / "set1000" / "known" / "candidates"

REGISTRY = json.load(open(os.path.join(ROOT, "data", "diseases.json"), encoding="utf-8"))
print("登録疾患:", {k: v["name"] for k, v in REGISTRY.items()})
D = REGISTRY[DISEASE_KEY]
DISEASE = D["name"]
DISEASE_INFO = D["info"][:5]
GENE_FILE = os.path.join(ROOT, "data", "genes", f"{D['gene_prefix']}_{GENE_SET}.tsv")
print("選択:", DISEASE, "| 遺伝子リスト:", os.path.basename(GENE_FILE))
for b in DISEASE_INFO: print("  -", b[:110] + ("…" if len(b) > 110 else ""))

登録疾患: {'ra': 'rheumatoid arthritis', 'scz': 'schizophrenia', 'cystinuria': 'cystinuria', 'prostate_cancer': 'prostate cancer', 'achondroplasia': 'achondroplasia'}
選択: rheumatoid arthritis | 遺伝子リスト: ra_set100.tsv
  - Joint pain, swelling and morning stiffness, symmetric in the small joints of hands and feet: caused by chronic…
  - Progressive joint deformity and loss of function: invasive growth of the synovial lining cells and excessive b…
  - The inflammation is sustained by the adaptive immune system: self-reactive T cells, antibody-producing B cells…
  - Fatigue, low-grade fever and anaemia: systemic effects of inflammatory mediators released by the activated imm…
  - Accelerated cardiovascular disease and interstitial lung disease: consequences of long-standing systemic infla…


## 設定

### このセルがすること：速度に関わる変数とモデルの選択を宣言する
- `BOTH_ORDERS`：全遺伝子を両順（「Yes or No」「No or Yes」）で聞き、対数オッズを平均して順序の癖を打ち消す（検証と同じ方式。1問なので両順でも速い）。`False` にすると yes_first だけ聞いて2倍速くなりますが、順序の癖（δ）が残ります。
- モデル：`~/llm/models` の GGUF（本命）か Ollama。`BACKEND` と `MODEL_SELECT` で選びます。

In [3]:
MAX_GENES = None                                   # 試運転なら 10
BOTH_ORDERS = True                                 # 全遺伝子を両順で聞く（検証と同じ）。False なら yes_first だけ
STRICT_PROMPT = True                               # 「大半の遺伝子は標的ではない」の前置き（検証はすべて True）
N_CTX = 2048

# --- モデルの選択 ---
BACKEND = "auto"                                   # "auto" / "gguf" / "ollama" / "mock"
MODEL_DIR = os.path.expanduser("~/llm/models")
OLLAMA_URL = "http://localhost:11434"
MODEL_SELECT = "auto"                              # "auto" / 一覧の番号 / 名前の一部
PREFER = ("txgemma", "medgemma", "gemma")
models = []
for h in sorted(glob.glob(os.path.join(MODEL_DIR, "**", "*.gguf"), recursive=True)):
    models.append({"kind": "gguf", "name": os.path.basename(h), "path": h, "size_gb": round(os.path.getsize(h) / 1e9, 2)})
try:
    with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=3) as r:
        for m in json.load(r).get("models", []):
            models.append({"kind": "ollama", "name": m["name"], "path": m["name"], "size_gb": round(m.get("size", 0) / 1e9, 2)})
except Exception:
    pass
chosen = None
if BACKEND != "mock":
    if isinstance(MODEL_SELECT, int):
        chosen = models[MODEL_SELECT]
    else:
        pool = [m for m in models if BACKEND == "auto" or m["kind"] == BACKEND]
        if MODEL_SELECT != "auto": pool = [m for m in pool if MODEL_SELECT.lower() in m["name"].lower()]
        ranked = sorted(pool, key=lambda m: (min([i for i, p in enumerate(PREFER) if p in m["name"].lower()] or [99]),
                                             0 if m["kind"] == "gguf" else 1, m["name"]))   # 高速版は GGUF を優先
        chosen = ranked[0] if ranked else None
USE_LLM = chosen is not None
BACKEND_USED = chosen["kind"] if chosen else "mock"
MODEL_PATH = chosen["path"] if chosen else None
ORDERS = ("yes_first", "no_first") if BOTH_ORDERS else ("yes_first",)
print("使えるモデル:"); [print(f"  [{i}] {m['kind']:6s} {m['size_gb']:6.2f} GB  {m['name']}") for i, m in enumerate(models)]
print("選択:", f"{BACKEND_USED}: {MODEL_PATH}" if USE_LLM else "モック（擬似乱数。数値に意味なし）", "| 順序:", ORDERS)
OUT_DIR = os.path.join(ROOT, "outputs"); os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, os.path.basename(GENE_FILE).replace(".tsv", "") + "_m2.csv")

使えるモデル:
  [0] gguf     7.59 GB  txgemma-9b-chat-Q6_K.gguf
  [1] ollama   7.59 GB  txgemma-9b-chat-q6_k:latest
  [2] ollama  17.40 GB  gemma3:27b
  [3] ollama   4.68 GB  qwen2.5:7b
  [4] ollama   1.16 GB  bge-m3:latest
  [5] ollama   9.28 GB  qwen3:14b
  [6] ollama   4.37 GB  cniongolo/biomistral:latest
  [7] ollama   4.01 GB  gemma3:4b-it-qat
  [8] ollama   5.23 GB  deepseek-r1:8b
  [9] ollama   5.23 GB  qwen3:8b
  [10] ollama   8.99 GB  qwen2.5:14b
  [11] ollama   4.92 GB  llama3.1:latest
選択: gguf: /Users/yoshinorisatomi/llm/models/txgemma-9b-chat-Q6_K.gguf | 順序: ('yes_first', 'no_first')


## 質問 M2 と、キャッシュが効くプロンプトの並び

### このセルがすること：質問を定義し、「前置き（全遺伝子で共通）」と「遺伝子ごとの部分」を分けて組み立てる
- 前置きには役割・厳しめの注意・病名・症状の箇条書き・答え方の指示まで入れます。**ここは1文字も変わらない**ので、1回処理して保存できます。
- 遺伝子ごとの部分は「Gene: 記号 (タンパク質名)」と M2 の文です。末尾の「Answer:」の位置で Yes/No の確率を読みます。
- 順序入れ替えは、前置きの指示文「Answer each question with Yes or No」/「… No or Yes」で行います（前置きが2通り → 保存する状態も2通り）。
- 文面は `scripts/yesno_question_variants.py` の `M2` と1文字も違わないようにしてあります（検証結果をそのまま当てはめるため）。変えたら検証し直してください。

In [4]:
QID = "M2"
QUESTION_M2 = ("Consider these three criteria:\n"
               "(a) A hypothesis can be constructed that inhibiting or activating the gene {gene} would treat {disease} or improve at least "
               "one of the symptoms listed above.\n"
               "(b) Even if this gene is not part of the pathway that causes {disease}, activating or inhibiting it could counteract or compensate "
               "for the abnormal process described above, for example through a parallel or opposing pathway in the same cells.\n"
               "(c) This gene's own specific role — its substrate, ligand, signalling pathway, cell type or circuit — matches the mechanism described "
               "above precisely, rather than a related but distinct one (e.g. a different molecule, cell type, tissue or subcellular compartment); "
               "a similar-sounding but distinct role, even in the same gene family, does not count.\n"
               "Does this gene meet at least one of these criteria?")

STRICT_NOTE = ("Note: the vast majority of human genes are NOT drug targets for any given disease. "
               "Answer Yes only when there is clear evidence or a clear mechanistic link; otherwise answer No. "
               "Membership in the same gene family, superfamily or protein class as a true disease gene (e.g. being "
               "another member of the same transporter, channel, receptor or enzyme family) is NOT sufficient evidence "
               "by itself. Judge each gene on whether ITS OWN specific substrate, ligand, cargo or interaction partner "
               "matches the mechanism described above, not on family resemblance alone.\n")

def prefix_text(order="yes_first"):
    """全遺伝子で共通の前置き。order で答え方の指示だけが変わる。"""
    info = "\n".join(f"- {b}" for b in DISEASE_INFO)
    options = "Answer each question with Yes or No." if order == "yes_first" else "Answer each question with No or Yes."
    return ("You are an expert in drug discovery and human disease biology.\n" + (STRICT_NOTE if STRICT_PROMPT else "") +
            f"Disease: {DISEASE}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{info}\n"
            f"{options}\n\n")

def gene_block(gene_label):
    return f"Gene: {gene_label}\n"

def question_line(symbol):
    return f"{QID}. {QUESTION_M2.format(disease=DISEASE, gene=symbol)} Answer:"

print(prefix_text() + gene_block("TNF (tumor necrosis factor)") + question_line("TNF"))

You are an expert in drug discovery and human disease biology.
Note: the vast majority of human genes are NOT drug targets for any given disease. Answer Yes only when there is clear evidence or a clear mechanistic link; otherwise answer No. Membership in the same gene family, superfamily or protein class as a true disease gene (e.g. being another member of the same transporter, channel, receptor or enzyme family) is NOT sufficient evidence by itself. Judge each gene on whether ITS OWN specific substrate, ligand, cargo or interaction partner matches the mechanism described above, not on family resemblance alone.
Disease: rheumatoid arthritis
Target symptoms and the organ, cell and functional abnormalities behind them:
- Joint pain, swelling and morning stiffness, symmetric in the small joints of hands and feet: caused by chronic inflammation of the synovial membrane that lines the joints
- Progressive joint deformity and loss of function: invasive growth of the synovial lining cells and

## 入力遺伝子

### このセルがすること：遺伝子リストを読み、プロンプトに入れる名前（記号＋タンパク質名）を作る
- タンパク質名は、ファイルの `protein_name_uniprot` → HGNC の `gene_name` → 記号のみ、の順で使います（UniProt の照会はノートブック 02 参照）。

In [5]:
genes = pd.read_csv(GENE_FILE, sep="\t", dtype=str).fillna("")
for col in ("protein_name_uniprot", "gene_name", "category", "label"):
    if col not in genes.columns: genes[col] = ""
if MAX_GENES: genes = genes.head(MAX_GENES).copy()
genes["gene_label"] = [f"{g['symbol']} ({(g['protein_name_uniprot'] or g['gene_name']).split('|')[0]})" if (g["protein_name_uniprot"] or g["gene_name"]) else g["symbol"] for _, g in genes.iterrows()]
print(len(genes), "genes"); genes[["symbol", "gene_label", "category"]].head(5)

100 genes


,symbol,gene_label,category
0,PADI4,PADI4 (peptidyl arginine deiminase 4),candidate
1,CCL21,CCL21 (C-C motif chemokine ligand 21),candidate
2,TNFRSF14,TNFRSF14 (TNF receptor superfamily member 14),candidate
3,IL7R,IL7R (interleukin 7 receptor),candidate
4,IRF5,IRF5 (interferon regulatory factor 5),candidate


## 高速エンジン（GGUF を llama-cpp-python で直接動かす）

### `last_logprobs()` — 直前に処理した位置の対数確率（全語彙）
どんな def か：llama.cpp のコンテキストから最後の位置の logits を読み、log-softmax にして返します。`llm.scores` は使いません（llama-cpp-python 0.3 系は `logits_all=False` だと `scores` を埋めないため）。
return：numpy 配列（語彙数）。

### `option_logprob(lp, ids)` — 綴り違いを合算した対数確率
どんな def か：「Yes」「 Yes」「yes」… の各トークン id の対数確率を合算（log-sum-exp）します。完全一致を優先すると末端の綴りを拾って逆転するため（ノートブック 02 で実機確認）。
return：float。

### `score_gene(gene_label, symbol, order)` — 1遺伝子を M2 で採点する
各ステップ：
1. 保存しておいた前置きの状態（`order` に応じて2通り）を、コンテキストを空にしてから `load_state` で復元する。前置きは再処理しない。
2. 遺伝子ブロックと M2 の文を処理する。
3. 「Answer:」の位置で Yes/No の確率を読む。
return：p_yes（float）。

### このセルがすること：エンジンを読み込み、前置きを2通り処理して状態を保存し、上の def を定義する
- Ollama では `/api/generate` → `/v1/completions` の順に logprobs 対応を試し、対応が確認できたエンドポイントを覚えて以後使い回します。**どちらも非対応なら、温度ありで8回サンプリングした Yes 割合で代用**します。

In [6]:
YES_SPELL, NO_SPELL = ["Yes", " Yes", "yes", " yes", "YES", " YES"], ["No", " No", "no", " no", "NO", " NO"]
prefix_state = {}

if BACKEND_USED == "gguf":
    from llama_cpp import Llama
    llm = Llama(model_path=MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=-1, logits_all=False, verbose=False)
    def tok(text, bos=False): return llm.tokenize(text.encode("utf-8"), add_bos=bos, special=bos)
    def ids_of(spellings):
        out = []
        for s in spellings:
            t = tok(s)
            if len(t) == 1 and t[0] not in out: out.append(t[0])
        return out
    YES_IDS, NO_IDS = ids_of(YES_SPELL), ids_of(NO_SPELL)
    for order in ORDERS:                                           # 前置きを順序ごとに1回だけ処理して保存
        llm.reset(); llm.eval(tok(prefix_text(order), bos=True))
        prefix_state[order] = (llm.save_state(), llm.n_tokens)
    print(f"GGUF engine ready | prefix tokens: {prefix_state['yes_first'][1]} | yes ids {YES_IDS} | no ids {NO_IDS}")
elif BACKEND_USED == "ollama":
    print("Ollama を使います（前置きのキャッシュ保存はできないため遅い経路です）:", MODEL_PATH)
else:
    print("警告: モデルが無いのでモック（擬似乱数）です。")

def last_logprobs():
    lg = np.ctypeslib.as_array(llm._ctx.get_logits(), shape=(llm.n_vocab(),)).astype(np.float64)
    lg = lg - lg.max()
    return lg - math.log(np.exp(lg).sum())

def option_logprob(lp, ids):
    v = [lp[i] for i in ids]; m = max(v)
    return m + math.log(sum(math.exp(x - m) for x in v))

def score_gene_gguf(gene_label, symbol, order="yes_first"):
    st, n = prefix_state[order]
    llm.reset()                                                     # 1a. 前回の遺伝子の続きを消す（残ったまま復元すると KV キャッシュが不整合になる）
    llm.load_state(st)                                              # 1b. 前置きの状態を復元
    llm.eval(tok(gene_block(gene_label) + question_line(symbol)))   # 2. 遺伝子ブロック + M2 → 「Answer:」の位置
    lp = last_logprobs()                                            # 3. Yes/No の確率
    ly, ln = option_logprob(lp, YES_IDS), option_logprob(lp, NO_IDS)
    return 1 / (1 + math.exp(ln - ly))

# ---- Ollama（遅い経路）: raw テンプレートで1回ずつ ----
def ollama_template(prompt_body):
    n = str(MODEL_PATH).lower()
    if "gemma" in n:        tpl = "<start_of_turn>user\n{body}<end_of_turn>\n<start_of_turn>model\nAnswer:"
    elif "qwen3" in n:      tpl = "<|im_start|>user\n{body}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nAnswer:"   # 思考ブロックを空にして答えの位置に来させる
    elif "qwen" in n or "deepseek" in n: tpl = "<|im_start|>user\n{body}<|im_end|>\n<|im_start|>assistant\nAnswer:"
    elif "mistral" in n:    tpl = "[INST] {body} [/INST] Answer:"
    elif "llama" in n:      tpl = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{body}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAnswer:"
    else:                   tpl = "{body}\nAnswer:"
    return tpl.format(body=prompt_body)

OLLAMA_LP_ENDPOINT = None   # 対応が確認できたエンドポイント（/api/generate か /v1/completions）。最初に一度だけ探す

def ollama_post(path, body):
    req = urllib.request.Request(OLLAMA_URL + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)

def ollama_top_logprobs(full_prompt, k=20):
    """先頭1トークンの上位 k 個の {token: log確率}。両エンドポイントとも非対応なら None。"""
    global OLLAMA_LP_ENDPOINT
    tries = [OLLAMA_LP_ENDPOINT] if OLLAMA_LP_ENDPOINT else ["/api/generate", "/v1/completions"]
    for ep in tries:
        try:
            if ep == "/api/generate":
                out = ollama_post(ep, {"model": MODEL_PATH, "prompt": full_prompt, "raw": True, "stream": False,
                                       "logprobs": True, "top_logprobs": k, "options": {"temperature": 0, "num_predict": 1}})
                lps = out.get("logprobs") or []
                top = {t["token"]: t["logprob"] for t in (lps[0].get("top_logprobs", []) if lps else [])}
                if lps and not top: top = {lps[0]["token"]: lps[0]["logprob"]}
            else:   # /v1/completions では logprobs は「個数」を渡す（真偽値ではない）
                ch = ollama_post(ep, {"model": MODEL_PATH, "prompt": full_prompt, "max_tokens": 1, "temperature": 0,
                                      "logprobs": k})["choices"][0]
                lp = ch.get("logprobs") or {}
                if lp.get("content"): top = {t["token"]: t["logprob"] for t in lp["content"][0].get("top_logprobs", [])}
                elif lp.get("top_logprobs"): top = dict(lp["top_logprobs"][0])
                else: top = {}
            if top:
                OLLAMA_LP_ENDPOINT = ep
                return top
        except Exception:
            continue
    return None

def ollama_generate_word(prompt_body, temperature=1.0):
    """logprobs が読めないときの代用。温度ありで1語だけ生成して Yes/No を読む。"""
    out = ollama_post("/api/generate", {"model": MODEL_PATH, "prompt": ollama_template(prompt_body), "raw": True,
                                        "stream": False, "options": {"temperature": temperature, "num_predict": 3}})
    m = re.match(r"\s*([A-Za-z]+)", out.get("response") or "")
    return m.group(1).lower() if m else ""

def score_gene_ollama(gene_label, symbol, order="yes_first"):
    body = prefix_text(order) + gene_block(gene_label) + question_line(symbol).replace(" Answer:", "")
    top = ollama_top_logprobs(ollama_template(body), 20)
    if top is None:                                                 # logprobs 非対応 → 8回サンプリングで代用（粗いが動く）
        yes = sum(1 for _ in range(8) if ollama_generate_word(body).startswith("yes"))
        return (yes + 0.5) / 9
    def lse(vals): m = max(vals); return m + math.log(sum(math.exp(v - m) for v in vals))
    y = [v for t, v in top.items() if t.strip().lower() == "yes"]; n = [v for t, v in top.items() if t.strip().lower() == "no"]
    floor = min(top.values())
    ly, ln = (lse(y) if y else floor), (lse(n) if n else floor)
    return 1 / (1 + math.exp(ln - ly))

def score_gene_mock(gene_label, symbol, order="yes_first"):
    return int(hashlib.md5((gene_label + QID + order).encode()).hexdigest(), 16) % 1000 / 1000

score_gene = {"gguf": score_gene_gguf, "ollama": score_gene_ollama}.get(BACKEND_USED, score_gene_mock)

if BACKEND_USED == "ollama":                                        # 最初に一度だけ logprobs 対応を確かめて表示する
    probe = ollama_top_logprobs(ollama_template("Answer with Yes or No.\nIs the sky blue?\nAnswer:"), 5)
    print(f"logprobs: 対応（{OLLAMA_LP_ENDPOINT}） 例 {dict(list(probe.items())[:3])}" if probe is not None else
          "logprobs: 非対応 → 8回サンプリングで代用します（p_yes は粗い推定値になります）")

t0 = time.time(); demo = {o: round(score_gene("TNF (tumor necrosis factor)", "TNF", o), 3) for o in ORDERS}; dt = time.time() - t0
print("test TNF:", demo, f"| {dt:.2f}s")

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


GGUF engine ready | prefix tokens: 287 | yes ids [3553, 6287, 3276, 7778, 17480, 21869] | no ids [1294, 1307, 956, 793, 5349, 6390]


test TNF: {'yes_first': 1.0, 'no_first': 1.0} | 2.47s


## 実行（1遺伝子 = M2 を1回 × 順序の数）

### このセルがすること：全遺伝子を採点して CSV に書く
- `p_yes_first` / `p_no_first`：各順序での「はい」確率。`delta`：その差（順序の癖）。
- `m2_logodds`：両順の対数オッズの平均（`BOTH_ORDERS=False` なら yes_first だけ）。**順位にはこれを使います**（飽和しない）。
- `p_m2`：`m2_logodds` を確率に戻したもの。**ダミーでも 0.5 を超えやすいので、絶対基準には使わないでください。**
- `rank`：`m2_logodds` の降順の順位（1 が最も標的らしい）。

In [7]:
logit = lambda x: math.log(max(x, 1e-6) / max(1 - x, 1e-6))
rows, t0 = [], time.time()
for i, (_, g) in enumerate(genes.iterrows()):
    p = {o: score_gene(g["gene_label"], g["symbol"], o) for o in ORDERS}
    lo = float(np.mean([logit(v) for v in p.values()]))
    rows.append({"symbol": g["symbol"], "gene_label": g["gene_label"], "category": g["category"], "label": g["label"],
                 **{f"p_{o}": round(v, 4) for o, v in p.items()},
                 "delta": round(p["yes_first"] - p["no_first"], 4) if BOTH_ORDERS else "",
                 "m2_logodds": round(lo, 3), "p_m2": round(1 / (1 + math.exp(-lo)), 4),
                 "model": f"{BACKEND_USED}:{os.path.basename(str(MODEL_PATH))}" if USE_LLM else "MOCK", "disease": DISEASE})
    if (i + 1) % 10 == 0 or i + 1 == len(genes):
        print(f"{i+1:4d}/{len(genes)} {g['symbol']:10s} {g['category']:9s} p_m2={rows[-1]['p_m2']:.3f} logodds={lo:+.2f}  ({time.time()-t0:.0f}s)")
res = pd.DataFrame(rows)
res["rank"] = res["m2_logodds"].rank(ascending=False, method="min").astype(int)
res = res.sort_values("rank").reset_index(drop=True)
res.to_csv(OUT_CSV, index=False)
print(f"elapsed {time.time()-t0:.0f}s for {len(genes)} genes → {(time.time()-t0)/len(genes):.2f}s/gene | wrote {OUT_CSV}")
if BOTH_ORDERS:
    d = res["delta"].astype(float).mean()
    print(f"順序の癖 δ（yes_first − no_first の平均）= {d:+.3f}" + ("  [注意] 並び順に敏感です" if abs(d) > 0.1 else ""))

  10/100 IKBKB      candidate p_m2=0.960 logodds=+3.17  (25s)


  20/100 IL23A      candidate p_m2=0.988 logodds=+4.44  (51s)


  30/100 IL17A      candidate p_m2=0.997 logodds=+5.89  (78s)


  40/100 IL10       candidate p_m2=0.995 logodds=+5.30  (104s)


  50/100 DHFR       known     p_m2=0.750 logodds=+1.10  (129s)


  60/100 BLK        candidate p_m2=0.571 logodds=+0.29  (156s)


  70/100 AFF3       candidate p_m2=0.331 logodds=-0.70  (182s)


  80/100 IL2RA      candidate p_m2=0.986 logodds=+4.24  (208s)


  90/100 CSF2RB     candidate p_m2=0.877 logodds=+1.97  (234s)


 100/100 POLGARF    random    p_m2=0.163 logodds=-1.63  (259s)
elapsed 260s for 100 genes → 2.60s/gene | wrote /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02/outputs/ra_set100_m2.csv
順序の癖 δ（yes_first − no_first の平均）= +0.316  [注意] 並び順に敏感です


## 評価

### このセルがすること：AUC と分類別の分布、上位の遺伝子を出す
- `AUC known vs others`：既知を候補・ダミーより上に置けるか。`known vs random`：既知をダミーより上に。`candidate vs random`：候補をダミーより上に（新規候補を拾う力）。
- **ファミリーの後光効果チェック**：ダミーを「正解と同じ遺伝子ファミリー」と「違うファミリー」に分け、それぞれで known との AUC を比べます。同じファミリーだけ AUC が低ければ、記号の見た目（SLC という名前など）に釣られている疑いがあります。
- 既知遺伝子の順位を一覧にします。下位にある既知遺伝子は、質問では拾えないタイプ（説明文の機序と違う経路で効く標的など）の可能性があります。

In [8]:
def auc(pos, neg):
    pos, neg = list(pos), list(neg)
    if not pos or not neg: return float("nan")
    return sum(1.0 if a > b else 0.5 if a == b else 0.0 for a in pos for b in neg) / (len(pos) * len(neg))
if not USE_LLM: print("警告: モックの数値です。")
known, other, rnd, cand = (res["category"] == "known"), (res["category"] != "known"), (res["category"] == "random"), (res["category"] == "candidate")
s = res["m2_logodds"]
print(f"AUC known vs others {auc(s[known], s[other]):.3f} | known vs random {auc(s[known], s[rnd]):.3f} | candidate vs random {auc(s[cand], s[rnd]):.3f}")
display(res.groupby("category")[["p_m2", "m2_logodds"]].median().round(3).rename(columns=lambda c: c + "（中央値）"))
print(f"ダミーの p_m2 中央値 = {res.loc[rnd, 'p_m2'].median():.2f}  ← 0.5 を超えていても異常ではない（M2 は OR なので Yes が出やすい）。順位で判断する")

# --- ファミリーの後光効果チェック ---
# 記号の「文字列+数字」部分をファミリーとみなす（例 SLC7A9 -> SLC7）。簡易的なヒューリスティックです。
_fam = lambda sym: (re.match(r"^([A-Za-z]+\d+)", sym).group(1) if re.match(r"^([A-Za-z]+\d+)", sym) else sym)
res["family"] = res["symbol"].apply(_fam)
res["same_family_as_known"] = res["family"].isin(set(res.loc[known, "family"]))
rnd_same, rnd_diff = rnd & res["same_family_as_known"], rnd & ~res["same_family_as_known"]
print(f"ダミーの内訳: 正解と同じファミリー {int(rnd_same.sum())} 件 / 違うファミリー {int(rnd_diff.sum())} 件")
if rnd_same.sum() >= 3 and rnd_diff.sum() >= 3:
    print(f"AUC known vs random（同じファミリー） {auc(s[known], s[rnd_same]):.3f} | （違うファミリー） {auc(s[known], s[rnd_diff]):.3f}"
          "  ← 同じファミリーだけ低ければ、ファミリー名に釣られている疑い")
else:
    print("同じ/違うファミリーのダミーがどちらも少なく（3件未満）、この比較は統計的に頼りません。")

print("\n既知遺伝子の順位（100 中）:")
display(res.loc[known, ["rank", "symbol", "gene_label", "p_m2", "m2_logodds"]])
print("上位 15:")
res.head(15)[["rank", "symbol", "category", "p_m2", "m2_logodds"] + [f"p_{o}" for o in ORDERS]]

AUC known vs others 0.719 | known vs random 0.976 | candidate vs random 0.940


,p_m2（中央値）,m2_logodds（中央値）
category,,
candidate,0.927,2.540
known,0.988,4.408
random,0.333,-0.696


ダミーの p_m2 中央値 = 0.33  ← 0.5 を超えていても異常ではない（M2 は OR なので Yes が出やすい）。順位で判断する
ダミーの内訳: 正解と同じファミリー 0 件 / 違うファミリー 14 件
同じ/違うファミリーのダミーがどちらも少なく（3件未満）、この比較は統計的に頼りません。

既知遺伝子の順位（100 中）:


,rank,symbol,gene_label,p_m2,m2_logodds
0,1,TNF,TNF (tumor necrosis factor),0.9999,9.514
1,2,IL6,IL6 (interleukin 6),0.9997,8.293
3,4,IL6R,IL6R (interleukin 6 receptor),0.9992,7.121
4,5,IL1R1,IL1R1 (interleukin 1 receptor type 1),0.9989,6.837
6,7,JAK1,JAK1 (Janus kinase 1),0.9973,5.909
12,13,JAK2,JAK2 (Janus kinase 2),0.9955,5.410
16,17,PTGS2,PTGS2 (prostaglandin-endoperoxide synthase 2),0.9933,5.004
20,21,JAK3,JAK3 (Janus kinase 3),0.9880,4.408
30,31,TNFSF11,TNFSF11 (TNF superfamily member 11),0.9713,3.522
48,49,NR3C1,NR3C1 (nuclear receptor subfamily 3 group C member 1),0.9184,2.421


上位 15:


,rank,symbol,category,p_m2,m2_logodds,p_yes_first,p_no_first
0,1,TNF,known,0.9999,9.514,1.0000,0.9998
1,2,IL6,known,0.9997,8.293,1.0000,0.9985
2,3,IL1B,candidate,0.9996,7.881,0.9999,0.9980
3,4,IL6R,known,0.9992,7.121,0.9999,0.9956
4,5,IL1R1,known,0.9989,6.837,0.9999,0.9867
5,6,TNFRSF1A,candidate,0.9982,6.323,0.9997,0.9882
6,7,JAK1,known,0.9973,5.909,0.9996,0.9829
7,8,IL17A,candidate,0.9973,5.894,0.9998,0.9692
8,9,CD40,candidate,0.9970,5.804,0.9994,0.9861
9,10,IL17RA,candidate,0.9962,5.575,0.9997,0.9486


## 対話型グラフ（ホバーで遺伝子名）

### このセルがすること：分類ごとの分布、順位付きの全遺伝子図、順序の癖の図を描く
- 各点にマウスを乗せると **記号・タンパク質名・分類・値** が出ます。凡例をクリックすると分類の表示／非表示を切り替えられます。
- 色と点の形の両方で分類を示します（青○ known、橙□ candidate、緑◇ random、黄▲ 正解と同じファミリーのダミー）。
- 同じ図を `outputs/<セット名>_m2_charts.html` にも保存します（ブラウザで開けば同じホバーが使えます）。
- `pip install plotly nbformat ipython` が必要です（ノートブック内に描くには nbformat が要ります。入れた後はカーネルを再起動）。

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CAT = [("known", "#2a78d6", "circle", "known"), ("candidate", "#eb6834", "square", "candidate"),
       ("random_other", "#1baf7a", "diamond", "random"), ("random_same", "#eda100", "triangle-up", "random（正解と同じファミリー）")]
MASK = {"known": known, "candidate": cand, "random_other": rnd & ~res["same_family_as_known"], "random_same": rnd & res["same_family_as_known"]}
CAT = [c for c in CAT if MASK[c[0]].any()]
hover = "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>%{customdata[2]}<br>順位 %{customdata[3]}<br>%{y:.3f}<extra></extra>"
cd = lambda d: d[["symbol", "gene_label", "category", "rank"]].values
rng_j = random.Random(0)

# --- 1. 分類ごとの分布（対数オッズ / 確率） ---
fig = make_subplots(rows=1, cols=2, subplot_titles=("m2_logodds（順位に使う）", "p_m2（確率。ダミーでも高めに出る）"))
for c, col in enumerate(("m2_logodds", "p_m2"), start=1):
    for j, (cat, color, sym, name) in enumerate(CAT):
        d = res[MASK[cat]]
        fig.add_trace(go.Scatter(x=[j + (rng_j.random() - 0.5) * 0.5 for _ in range(len(d))], y=d[col], mode="markers",
                                 name=name, legendgroup=cat, showlegend=(c == 1),
                                 marker=dict(color=color, symbol=sym, size=9 if cat != "candidate" else 7, opacity=0.8, line=dict(width=1, color="#fcfcfb")),
                                 customdata=cd(d), hovertemplate=hover), row=1, col=c)
    fig.update_xaxes(tickvals=list(range(len(CAT))), ticktext=[x[3].split("（")[0] for x in CAT], row=1, col=c)
fig.add_hline(y=0.5, line=dict(color="#c3c2b7", dash="dot"), row=1, col=2)
fig.update_layout(height=440, width=1100, title=f"{DISEASE}: M2 score by category (hover = gene)", template="plotly_white")
fig.show()

# --- 2. 順位付きの全遺伝子 ---
fig2 = go.Figure()
for cat, color, sym, name in CAT:
    d = res[MASK[cat]]
    fig2.add_trace(go.Scatter(x=d["rank"], y=d["m2_logodds"], mode="markers", name=name,
                              marker=dict(color=color, symbol=sym, size=9, line=dict(width=1, color="#fcfcfb")),
                              customdata=cd(d), hovertemplate=hover))
kn = res[known]
fig2.add_trace(go.Scatter(x=kn["rank"], y=kn["m2_logodds"], mode="text", text=kn["symbol"], textposition="top center",
                          textfont=dict(size=10, color="#2a78d6"), showlegend=False, hoverinfo="skip"))
fig2.update_layout(title=f"{DISEASE}: all genes ranked by M2 (known genes labelled)", xaxis_title="rank", yaxis_title="m2_logodds",
                   template="plotly_white", width=1100, height=440)
fig2.show()

figs = [fig, fig2]
# --- 3. 順序の癖（yes_first × no_first）。対角線から外れるほど並び順で答えが変わる ---
if BOTH_ORDERS:
    fig3 = go.Figure()
    for cat, color, sym, name in CAT:
        d = res[MASK[cat]]
        fig3.add_trace(go.Scatter(x=d["p_no_first"], y=d["p_yes_first"], mode="markers", name=name,
                                  marker=dict(color=color, symbol=sym, size=8, line=dict(width=1, color="#fcfcfb")),
                                  customdata=cd(d), hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[2]}<br>no_first %{x:.3f}<br>yes_first %{y:.3f}<extra></extra>"))
    fig3.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(color="#c3c2b7", dash="dot"))
    fig3.update_layout(title=f"{DISEASE}: order bias — p_yes (Yes-or-No) vs p_yes (No-or-Yes)", xaxis_title="p_no_first", yaxis_title="p_yes_first",
                       template="plotly_white", width=560, height=520, xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]))
    fig3.show(); figs.append(fig3)

html_path = OUT_CSV.replace(".csv", "_charts.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'></head><body>")
    for i, fg in enumerate(figs):
        f.write(fg.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False))
    f.write("</body></html>")
print("saved:", html_path)

saved: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02/outputs/ra_set100_m2_charts.html
